# Week 4: Backpropagation, SGD, and Regularization

**Lecture 7:** Backpropagation implementation  
**Lecture 8:** Stochastic gradient descent and regularization

# Lecture 7: Implementing Backpropagation

This implementation follows the equations from Week 3. Hidden layers use sigmoid. Setting `loss="cce"` selects a softmax output for classification; `loss="mse"` selects a linear output for regression. The backward pass uses the matching output delta, propagates it backward, and constructs explicit gradients for each $W^{(\ell)}$ and $b^{(\ell)}$.

Lecture 7 deliberately uses **full-batch gradient descent**: every update is computed from the complete training set. Lecture 8 introduces a self-contained mini-batch SGD class so the full training algorithm remains visible in one place.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
class MultilayerPerceptron:
    def __init__(self, layer_sizes, alpha=0.5, loss="cce", seed=1):
        if loss not in {"cce", "mse"}:
            raise ValueError("loss must be 'cce' or 'mse'")
        self.alpha = alpha
        self.loss_name = loss
        self.rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2 / (input_size + output_size))
            self.weights.append(
                self.rng.normal(0, scale, size=(input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def sigmoid_gradient(a):
        return a * (1 - a)

    @staticmethod
    def softmax(z):
        shifted_z = z - z.max(axis=1, keepdims=True)
        exp_z = np.exp(shifted_z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def forward(self, X):
        # Cache every layer activation because backpropagation reuses it.
        activations = [np.atleast_2d(X)]

        for layer, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = activations[-1] @ W + b
            if layer == len(self.weights) - 1:
                # CCE uses softmax; MSE regression uses a linear output.
                a = self.softmax(z) if self.loss_name == "cce" else z
            else:
                a = self.sigmoid(z)
            activations.append(a)

        return activations

    def loss(self, X, y):
        predictions = self.forward(X)[-1]
        if self.loss_name == "cce":
            probabilities = np.clip(predictions, 1e-12, 1.0)
            return -np.mean(np.sum(y * np.log(probabilities), axis=1))

        # Average the summed squared output error across examples.
        return np.mean(np.sum((predictions - y) ** 2, axis=1))

    def gradients(self, X, y):
        X = np.atleast_2d(X)
        y = np.atleast_2d(y)
        n = X.shape[0]
        activations = self.forward(X)

        weight_gradients = [None] * len(self.weights)
        bias_gradients = [None] * len(self.biases)

        if self.loss_name == "cce":
            # Softmax plus CCE gives this output delta.
            delta = activations[-1] - y
        else:
            # The MSE output is linear, so its activation derivative is one.
            delta = 2 * (activations[-1] - y)

        for layer in range(len(self.weights) - 1, -1, -1):
            # Connect the previous activation to each unit's current error.
            weight_gradients[layer] = activations[layer].T @ delta / n
            bias_gradients[layer] = delta.mean(axis=0)

            if layer > 0:
                # Move the error through W, then through sigmoid's derivative.
                delta = (
                    delta @ self.weights[layer].T
                    * self.sigmoid_gradient(activations[layer])
                )

        return weight_gradients, bias_gradients

    def step(self, X, y):
        weight_gradients, bias_gradients = self.gradients(X, y)

        # Update every layer only after all of its gradients are available.
        for layer in range(len(self.weights)):
            self.weights[layer] -= self.alpha * weight_gradients[layer]
            self.biases[layer] -= self.alpha * bias_gradients[layer]

    def accuracy(self, X, y):
        if self.loss_name != "cce":
            raise ValueError("accuracy is only defined for CCE classification")
        targets = y.argmax(axis=1) if y.ndim == 2 else y
        return np.mean(self.predict(X) == targets)

    def print_history_header(self):
        if self.loss_name == "cce":
            print(
                f"{'epoch':>7} {'loss':>12} {'val_loss':>12} "
                f"{'accuracy':>12} {'val_accuracy':>14}"
            )
            print("-" * 63)
        else:
            print(f"{'epoch':>7} {'loss':>12} {'val_loss':>12}")
            print("-" * 35)

    def record_metrics(self, epoch, X, y, validation_data):
        X_val, y_val = validation_data
        metrics = {
            "epoch": epoch,
            "loss": self.loss(X, y),
            "val_loss": self.loss(X_val, y_val),
        }
        if self.loss_name == "cce":
            metrics["accuracy"] = self.accuracy(X, y)
            metrics["val_accuracy"] = self.accuracy(X_val, y_val)
            print(
                f"{metrics['epoch']:7d} {metrics['loss']:12.6f} "
                f"{metrics['val_loss']:12.6f} {metrics['accuracy']:12.4f} "
                f"{metrics['val_accuracy']:14.4f}"
            )
        else:
            print(
                f"{metrics['epoch']:7d} {metrics['loss']:12.6f} "
                f"{metrics['val_loss']:12.6f}"
            )
        return metrics

    def fit(self, X, y, validation_data, epochs=100, update=10):
        history = []
        self.print_history_header()

        for epoch in range(epochs):
            # One update from the entire training set.
            self.step(X, y)

            if (epoch + 1) % update == 0 or epoch == 0:
                history.append(
                    self.record_metrics(epoch + 1, X, y, validation_data)
                )

        return history

    def predict(self, X):
        output = self.forward(X)[-1]
        return output.argmax(axis=1) if self.loss_name == "cce" else output

    def predict_proba(self, X):
        if self.loss_name != "cce":
            raise ValueError("predict_proba is only available for CCE")
        return self.forward(X)[-1]

### The Same Class for Regression

Selecting the MSE loss replaces softmax with a linear output. The same network can then predict one or more numerical outputs.

In [ ]:
rng = np.random.default_rng(1)
X_regression = np.linspace(-2, 2, 200).reshape(-1, 1)
y_regression = np.column_stack([
    1 + 2 * X_regression[:, 0],
    -0.5 + 0.75 * X_regression[:, 0],
])
y_regression += rng.normal(0, 0.05, size=y_regression.shape)

X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_regression, y_regression, test_size=0.25, random_state=1
)

regression_model = MultilayerPerceptron(
    [1, 8, 2], alpha=0.5, loss="mse", seed=1
)
regression_history = regression_model.fit(
    X_reg_train,
    y_reg_train,
    validation_data=(X_reg_val, y_reg_val),
    epochs=1_000,
    update=100,
)

print("first three predictions:")
print(np.round(regression_model.predict(X_reg_val[:3]), 3))
print("first three targets:")
print(np.round(y_reg_val[:3], 3))

### Tiny MNIST Example

Scikit-learn's digits dataset is a small, built-in analogue of MNIST. It contains 1,797 grayscale images at only $8\times8$ pixels, so we can inspect the full-batch implementation quickly before moving to $28\times28$ MNIST.

In [ ]:
digits = load_digits()
X_tiny = digits.data / 16.0
y_tiny = np.eye(10)[digits.target]

X_tiny_train, X_tiny_test, y_tiny_train, y_tiny_test = train_test_split(
    X_tiny,
    y_tiny,
    test_size=0.25,
    random_state=1,
    stratify=digits.target,
)

tiny_model = MultilayerPerceptron(
    [64, 32, 10], alpha=1.0, loss="cce", seed=1
)
tiny_history = tiny_model.fit(
    X_tiny_train,
    y_tiny_train,
    validation_data=(X_tiny_test, y_tiny_test),
    epochs=1_000,
    update=100,
)
print(
    classification_report(
        y_tiny_test.argmax(axis=1), tiny_model.predict(X_tiny_test)
    )
)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
predictions = tiny_model.predict(X_tiny_test[:10])

for image, prediction, target, ax in zip(
    X_tiny_test[:10],
    predictions,
    y_tiny_test[:10].argmax(axis=1),
    axes.ravel(),
):
    ax.imshow(image.reshape(8, 8), cmap="gray")
    ax.set_title(f"pred={prediction}, true={target}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### MNIST with 1,000 Training Images

MNIST contains $28\times28$ images, giving 784 input features. To expose the cost and limitations of full-batch learning, Lecture 7 trains on only the first 1,000 training images. The test set remains separate.

In [ ]:
with np.load('../../datasets/mnist.npz') as mnist_data:
    X_mnist_train_full = mnist_data['x_train']
    y_mnist_train_full = mnist_data['y_train']
    X_mnist_test = mnist_data['x_test']
    y_mnist_test = mnist_data['y_test']

X_mnist_train = (
    X_mnist_train_full[:1_000].reshape(1_000, 28 * 28).astype("float32") / 255
)
y_mnist_train = np.eye(10)[y_mnist_train_full[:1_000]]

# A 2,000-image test subset keeps this instructional run quick.
X_mnist_test_small = (
    X_mnist_test[:2_000].reshape(2_000, 28 * 28).astype("float32") / 255
)
y_mnist_test_small = y_mnist_test[:2_000]

mnist_1k_model = MultilayerPerceptron(
    [784, 32, 10], alpha=1.0, loss="cce", seed=1
)
mnist_1k_history = mnist_1k_model.fit(
    X_mnist_train,
    y_mnist_train,
    validation_data=(
        X_mnist_test_small, np.eye(10)[y_mnist_test_small]
    ),
    epochs=200,
    update=20,
)
print(
    classification_report(
        y_mnist_test_small,
        mnist_1k_model.predict(X_mnist_test_small),
    )
)

# Lecture 8: Stochastic Gradient Descent and Regularization

Full-batch gradient descent computes one update from all $n$ examples. **Stochastic gradient descent (SGD)** estimates the gradient from a shuffled mini-batch $B$:

$$\nabla L_B(w)=\frac{1}{|B|}\sum_{i\in B}\nabla L_i(w).$$

The class below repeats the complete network implementation instead of hiding behavior through inheritance. It uses the same explicit backpropagation equations but makes many smaller updates per epoch.

It also supports L2 regularization (weight decay):

$$L_{\mathrm{regularized}}(w)=L_{\mathrm{data}}(w)+\frac{\lambda}{2}\sum_{\ell}\|W^{(\ell)}\|_F^2.$$

The explicit weight gradient therefore gains $\lambda W^{(\ell)}$. Biases are not penalized.

In [ ]:
class StochasticMultilayerPerceptron:
    def __init__(
        self,
        layer_sizes,
        alpha=0.1,
        batch_size=64,
        l2=0.0,
        loss="cce",
        seed=1,
    ):
        if loss not in {"cce", "mse"}:
            raise ValueError("loss must be 'cce' or 'mse'")

        self.alpha = alpha
        self.batch_size = batch_size
        self.l2 = l2
        self.loss_name = loss
        self.rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        # Build one weight matrix and bias vector for each affine layer.
        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2 / (input_size + output_size))
            self.weights.append(
                self.rng.normal(0, scale, size=(input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def sigmoid_gradient(a):
        # Reuse the stored sigmoid activation instead of recomputing sigmoid.
        return a * (1 - a)

    @staticmethod
    def softmax(z):
        shifted_z = z - z.max(axis=1, keepdims=True)
        exp_z = np.exp(shifted_z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def forward(self, X):
        # Cache every activation because the backward pass reuses them.
        activations = [np.atleast_2d(X)]

        for layer, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = activations[-1] @ W + b

            if layer == len(self.weights) - 1:
                # CCE is classification; MSE is regression/reconstruction.
                a = self.softmax(z) if self.loss_name == "cce" else z
            else:
                a = self.sigmoid(z)

            activations.append(a)

        return activations

    def loss(self, X, y):
        predictions = self.forward(X)[-1]

        if self.loss_name == "cce":
            probabilities = np.clip(predictions, 1e-12, 1.0)
            data_loss = -np.mean(
                np.sum(y * np.log(probabilities), axis=1)
            )
        else:
            data_loss = np.mean(
                np.sum((predictions - y) ** 2, axis=1)
            )

        # Penalize weights but not biases; 1/2 cancels during differentiation.
        penalty = 0.5 * self.l2 * sum(
            np.sum(W**2) for W in self.weights
        )
        return data_loss + penalty

    def gradients(self, X, y):
        X = np.atleast_2d(X)
        y = np.atleast_2d(y)
        n = X.shape[0]
        activations = self.forward(X)

        weight_gradients = [None] * len(self.weights)
        bias_gradients = [None] * len(self.biases)

        if self.loss_name == "cce":
            # Softmax plus CCE gives this output delta.
            delta = activations[-1] - y
        else:
            # The MSE output is linear, so its activation derivative is one.
            delta = 2 * (activations[-1] - y)

        for layer in range(len(self.weights) - 1, -1, -1):
            weight_gradients[layer] = (
                activations[layer].T @ delta / n
                + self.l2 * self.weights[layer]
            )
            bias_gradients[layer] = delta.mean(axis=0)

            if layer > 0:
                delta = (
                    delta @ self.weights[layer].T
                    * self.sigmoid_gradient(activations[layer])
                )

        return weight_gradients, bias_gradients

    def step(self, X, y):
        weight_gradients, bias_gradients = self.gradients(X, y)

        # Update only after every gradient has been calculated.
        for layer in range(len(self.weights)):
            self.weights[layer] -= self.alpha * weight_gradients[layer]
            self.biases[layer] -= self.alpha * bias_gradients[layer]

    def accuracy(self, X, y):
        if self.loss_name != "cce":
            raise ValueError("accuracy is only defined for CCE classification")
        targets = y.argmax(axis=1) if y.ndim == 2 else y
        return np.mean(self.predict(X) == targets)

    def print_history_header(self):
        if self.loss_name == "cce":
            print(
                f"{'epoch':>7} {'loss':>12} {'val_loss':>12} "
                f"{'accuracy':>12} {'val_accuracy':>14}"
            )
            print("-" * 63)
        else:
            print(f"{'epoch':>7} {'loss':>12} {'val_loss':>12}")
            print("-" * 35)

    def record_metrics(self, epoch, X, y, validation_data):
        X_val, y_val = validation_data
        metrics = {
            "epoch": epoch,
            "loss": self.loss(X, y),
            "val_loss": self.loss(X_val, y_val),
        }

        if self.loss_name == "cce":
            metrics["accuracy"] = self.accuracy(X, y)
            metrics["val_accuracy"] = self.accuracy(X_val, y_val)
            print(
                f"{metrics['epoch']:7d} {metrics['loss']:12.6f} "
                f"{metrics['val_loss']:12.6f} {metrics['accuracy']:12.4f} "
                f"{metrics['val_accuracy']:14.4f}"
            )
        else:
            print(
                f"{metrics['epoch']:7d} {metrics['loss']:12.6f} "
                f"{metrics['val_loss']:12.6f}"
            )

        return metrics

    def fit(self, X, y, validation_data, epochs=10, update=1):
        X = np.asarray(X)
        y = np.asarray(y)
        history = []
        self.print_history_header()

        for epoch in range(epochs):
            # Shuffle once per epoch before slicing consecutive mini-batches.
            indices = self.rng.permutation(len(X))

            for start in range(0, len(X), self.batch_size):
                batch = indices[start : start + self.batch_size]
                # Each batch performs its own forward, backward, and update.
                self.step(X[batch], y[batch])

            if (epoch + 1) % update == 0 or epoch == 0:
                history.append(
                    self.record_metrics(epoch + 1, X, y, validation_data)
                )

        return history

    def predict(self, X):
        output = self.forward(X)[-1]
        return output.argmax(axis=1) if self.loss_name == "cce" else output

    def predict_proba(self, X):
        if self.loss_name != "cce":
            raise ValueError("predict_proba is only available for CCE")
        return self.forward(X)[-1]

### Full MNIST with Mini-batch SGD

The SGD class now trains on all 60,000 MNIST training images. Each epoch uses every image once, but updates occur after each shuffled mini-batch rather than after the entire dataset.

In [ ]:
X_mnist_train_full = (
    X_mnist_train_full.reshape(60_000, 28 * 28).astype("float32") / 255
)
X_mnist_test_full = (
    X_mnist_test.reshape(10_000, 28 * 28).astype("float32") / 255
)
y_mnist_train_full_one_hot = np.eye(10)[y_mnist_train_full]

sgd_model = StochasticMultilayerPerceptron(
    [784, 32, 16, 10],
    alpha=0.5,
    batch_size=64,
    l2=1e-5,
    loss="cce",
    seed=1,
)
sgd_history = sgd_model.fit(
    X_mnist_train_full,
    y_mnist_train_full_one_hot,
    validation_data=(X_mnist_test_full, np.eye(10)[y_mnist_test]),
    epochs=10,
    update=1,
)
print(classification_report(y_mnist_test, sgd_model.predict(X_mnist_test_full)))

## Code Takeaways

- Backpropagation reuses cached forward-pass values and propagates output error backward one layer at a time.
- CCE uses a softmax output for classification; MSE uses a linear output for regression and reconstruction.
- Each weight gradient is an outer product of the previous activation and the current layer error; bias gradients sum the errors across examples.
- The tiny MNIST example makes individual tensors inspectable before the same implementation is applied to 1,000 examples.
- Mini-batch SGD updates parameters many times per epoch, unlike full-batch gradient descent.
- Shuffling prevents every epoch from presenting mini-batches in the same order.
- L2 regularization adds a weight-dependent term to both the loss and gradient, discouraging unnecessarily large weights.
- Printing training and validation metrics together makes optimization failure and overfitting distinguishable.